# AQPotency AWS Marketplace Sample Notebook

Clone-and-run sample notebook for the **AQPotency** SageMaker model package. This notebook guides you through set up, running concrete examples of the main features, and clean up.

> ⚠️ This notebook focuses on the concrete experience of the product. If you want to better understand how AWS SageMaker works, the pricing and how to subscribe to and launch this product using AWS Marketplace, please consult `onboarding_aws_sagemaker.md` at the top level of this repo.

---

AQPotency is a fast, cheap CPU ML model for potency prediction enabling drug-discovery teams to rank and prioritize the most promising small-molecule ligands for a given protein target, or identify the proteins most likely to interact with a given ligand. It can be used as a large-scale triage tool ahead of a virtual screening or reverse virtual screening workflow.

Researchers provide two core inputs — a protein target (UniProt ID) and a set of candidate molecules (SMILES) — and AQPotency returns ranked potency predictions for each protein-ligand pair, together with uncertainty and applicability signals that help teams tell stronger opportunities apart from lower-confidence results.

---

**Outline:**

1) Subscribe and set up your environment
2) Real-time inference: `potency`, `scan`, `screen` and `selectivity` (+ clean up)
3) Batch transform: `screen` at large scales — recommended for large library triage

> 💰 The notebook creates **billable** resources. Kill the endpoints when you finish. See pricing on AWS marketplace listing.
> For **real-time** operations, billing is proportional to the number of successful predictions.
> For **batch transform**, billing is pro-rated per machine hour (by the second).

> ⚠️ We recommend checking for any live endpoint using the AWS web console in SageMaker and terminating them when you are done. See our Sagemaker onboarding guide.

---

## 1. Subscribe and set up your environment

You can find this product among the ones sold by [SandboxAQ on AWS Marketplace](https://aws.amazon.com/marketplace/seller-profile?id=seller-yepgc2kakn3xo).

Consult `onboarding_aws_sagemaker.md` to subscribe to the product, understand at the pricing, and retrieve the arn corresponding to the version used and your compute region. It contains useful information and images. Copy-paste it in the cell below.

In [1]:
model_package_arn = ''  # paste your region ARN here

Regardless of your environment, you can retrieve this notebook and other relevant files by running `git clone https://github.com/sandbox-quantum/marketplace-public-docs.git` from a terminal, or a jupyter notebook if you add a `!` before the command.

### Set up option A — Amazon SageMaker Studio web UI (fastest)

- See our `onboarding_aws_sagemaker.md` guide.
- Once this notebook is open in SageMaker Studio web UI, you're ready.

### Set up option B — local / other environment
- Clone this repo into your environment and navigate to this directory
- Ensure you have the packages specified in [requirements.txt](./requirements.txt) installed in your environment. Otherwise, you can run:
    ```bash
    pip install -r requirements.txt
    aws configure          # credentials + default region
    ```

Then run the code below to set up the session.

> ⚠️ Make sure the region printed by the cell below indeed matches the one you used for the model arn.

In [2]:
import json
import boto3
import time
import pandas as pd
from pprint import pprint

# SageMaker SDK v3 first, fall back to v2.
try:
    from sagemaker.core.helper.session_helper import Session, get_execution_role  # SDK v3
except ImportError:
    from sagemaker import Session, get_execution_role                             # SDK v2

session = Session()
region  = session.boto_region_name
role    = get_execution_role()

sagemaker_client = boto3.client('sagemaker', region_name=region)
runtime          = boto3.client('sagemaker-runtime', region_name=region)
s3_client        = boto3.client('s3', region_name=region)

print('Region :', region)
print('Role   :', role)
print('Using model package:', model_package_arn)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region : us-east-1
Role   : arn:aws:iam::769953010191:role/service-role/AmazonSageMaker-ExecutionRole-20260729T220732
Using model package: arn:aws:sagemaker:us-east-1:865070037744:model-package/aqpotency-v1-0-8-2603a1cbadcc3f728179370c6751c12d


## 2. Real-time inference

[Real-time inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) is designed to supports "small" blocking requests between an end user and a live, persistent endpoint. This SageMaker feature comes with limitations (limited payload size for inputs/outputs, time out of a few minutes, no files saved). Usage is priced by successful prediction, not machine time. For large asynchronous requests with large inputs (> 500k SMILES), check out the batch transform section.

There are four modes available in real-time inference:`potency`, `scan`, `selectivity` and `screen`. This notebook describes each of these modes through examples. Before you start, some important information:

* All these **real-time inference** modes support limited input size per call. `potency` and `selectivity` are convenience features with low max input size while `scan` and `screen` are designed to run at larger scales. Any call with an input size higher than the allowed maximum returns an error and prints the cap. At the time of writing, the caps are:

| Mode | Max input size per call |
|-|-|
| `potency` | 15 |
| `selectivity` | 15 |
| `screen` | 10000 |

* In real-time inference, inputs and outputs are JSON payloads. This data is not saved anywhere unless you explicitly write it to file. 
* Proteins / targets are passed as **UniProt IDs**, while small molecule ligands are passed as **SMILES**.
* The model returns several scores:
  * a **potency prediction** `potency_mean`, binding affinity score on the pIC50 scale ($-\log_{10}(\text{IC}_{50}))$. A score $0.7<= m <=0.9$ implies a prediction within sub-micromolar to nanomolar range.
  * an **uncertainty score** (`potency_sigma`), which is the estimated standard deviation (σ) of the predictive distribution in pIC50 units for a given compound–protein pair. Because pIC50 is on a base-10 logarithmic scale, an error in pIC50 translates directly to a multiplicative (fold-change) error in IC50. $σ<=0.5$ log units is considered high confidence (up to ~3.2x fold-change in IC50), $0.5<=σ<=0.8$ log units is considered moderate confidence (3.2x-10x fold-change in IC50, typical for bioscreening activity). Values above imply the model sampled very different values and therefore the `potency_mean` is not reliable.
  * An optional **applicability_domain** ("AD", `applicability_domain`) score in interval `[0, 1]` indicates how much overlap the inputs have with the training set of AQPotency. A score $s=1$ means it was in the training set, and $s<=0.4$ can be considered unreliable / noise and could be discarded. We strongly recommend computing this score. You can however turn it off using the boolean option `applicability_domain: False` to make predictions faster.
* Only valid inputs will be scored and billed, invalid inputs (e.g invalid SMILES or UniProt IDs) will be ignored. Check out `n_scored` in the resturn JSON payload.

In order to run the real-time AQPotency examples below, we first need to create a SageMaker model.

In [ ]:
# Create the model from the AWS Marketplace model package, with network isolation enabled
model_name = 'aqpotency-sample-model'

sagemaker_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    Containers=[{'ModelPackageName': model_package_arn}],
    EnableNetworkIsolation=True,   # required for AWS Marketplace model packages
)
print('Created model:', model_name)

We then deploy a "real-time" endpoint - it may take several minutes. 
This endpoint needs to be terminated manually when you are done to stop the billing, using the section at the end of this notebook (or the AWS SageMaker console).

In [ ]:
# Deploy a real-time endpoint. This may take a few minutes.
endpoint_name        = 'aqpotency-sample-endpoint'
endpoint_config_name = 'aqpotency-sample-config'

sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': model_name,
        'InstanceType': 'ml.m5.2xlarge',
        'InitialInstanceCount': 1,
    }],
)
sagemaker_client.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print('Deploying endpoint — this takes a few minutes...')
sagemaker_client.get_waiter('endpoint_in_service').wait(EndpointName=endpoint_name)
print('Endpoint live:', endpoint_name)

We define a convenience function to invoke our real-time endpoint in our following examples:

In [3]:
def invoke(payload):
    r = runtime.invoke_endpoint(
        EndpointName=endpoint_name, ContentType='application/json',
        Accept='application/json', Body=json.dumps(payload))
    return json.loads(r['Body'].read())

### 2a. Potency (`mode:"potency"`)

Convenience feature intended to score a few arbitrary protein-ligand pairs. This mode does not cache protein or ligand information and is therefore slower than other modes, designed to handle larger volumes. The result comes as a JSON payload the user can parse like a python dictionary (we show convenient ways to turn such objects as pandas `DataFrame` or `csv` files in the next examples).

Please look at the schema file for this mode for comprehensive details.

In [7]:
potency_payload = {
    'mode': 'potency',                   # required
    'applicability_domain': True,        # set False to skip AD scores (score becomes null)
    'instances': [
        {'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1', 'uniprot_id': 'P00533'},
        {'smiles': 'CCO', 'uniprot_id': 'P11362'}
    ],
}
result = invoke(potency_payload)
print(json.dumps(result, indent=2))

# Parse output
p = result['predictions'][0]
print(f"\npIC50 = {p['potency_mean']} +/- {p['potency_sigma']} "
      f"(AD = {p['applicability_domain']}, target = {p['target_name']})")

{
  "predictions": [
    {
      "smiles": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
      "uniprot_id": "P00533",
      "potency_mean": 7.639,
      "potency_sigma": 1.115,
      "applicability_domain": 1.0,
      "target_name": "epidermal growth factor receptor"
    },
    {
      "smiles": "CCO",
      "uniprot_id": "P11362",
      "potency_mean": 6.192,
      "potency_sigma": 1.73,
      "applicability_domain": 0.135,
      "target_name": "fibroblast growth factor receptor 1"
    }
  ],
  "warnings": [
    "P11362: low applicability domain score (0.14)."
  ]
}

pIC50 = 7.639 +/- 1.115 (AD = 1.0, target = epidermal growth factor receptor)


The results need to be contextualized with the uncertainty and applicability domain scores. A `potency_sigma` > 1. or `applicability_domain` < 0.4 means the `potency_mean` is unreliable. We plan to improve the model reliability and accuracy in the future by gradually expanding the training set.

Real-time modes have a `warning` section in the payload for low applicability domain scores, although it is simpler to segregate / remove them programmatically using dataframes, as demonstrated in other examples below.

### 2b. Scan (`mode:"scan"`)

Compute potency predictions for a given ligand against one of the available pre-defined panels of targets:

- `bowes_safety_panel`: 46 proteins, general safety off-target panel to flag potential safety liabilities (e.g., hERG cardiac channel, key receptors, transporters)
- `human_kinome` 604 proteins, spanning the full human kinase space.
- `proteome`: 20,415 proteins, spanning the full human proteome. 

A `scan` against the `proteome` panel above takes about ~30s at the time of writing, while the smaller ones complete in 3-5s.

* Optional parameter `threshold` allow you to surface an `alert` boolean for targets whose potency prediction is over the desired value.
* Optional parameter `top_n` can be used to only return the `n` top scorers.

Please look at the schema file for this mode for comprehensive details.

In [8]:
scan_payload = {
    'mode': 'scan',
    'instances': [
        {'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
         'panel': 'bowes_safety_panel',
         'threshold': 7.0,
         #'top_n': 10 # uncomment to turn this parameter on
        },
    ],
}
result = invoke(scan_payload)

# Parse output
scan = result['predictions'][0]
print(f"Scanned {scan['n_targets']} targets in panel '{scan['panel']}\'\n\nPrinting a few rows of the result:")
pprint(scan['hits'][:2])

Scanned 45 targets in panel 'bowes_safety_panel'

Printing a few rows of the result:
[{'alert': True,
  'applicability_domain': 0.358,
  'gene_symbol': 'EDNRA',
  'potency_mean': 7.097,
  'potency_sigma': 1.064,
  'source': 'Bowes 2012',
  'target_class': 'GPCR',
  'target_name': 'endothelin-1 receptor',
  'uniprot_id': 'P25101'},
 {'alert': False,
  'applicability_domain': 0.298,
  'gene_symbol': 'ADORA2A',
  'potency_mean': 6.849,
  'potency_sigma': 0.675,
  'source': 'Bowes 2012',
  'target_class': 'GPCR',
  'target_name': 'adenosine receptor a2a',
  'uniprot_id': 'P29274'}]


Instead of using the JSON format for further processing the output data, pandas `DataFrame` and `csv` files are more convenient. They allow for quick visualization but also facilitate ranking and filtering - programmatically or through your favorite spreadsheet UI. It can be accomplished in a few lines:

In [29]:
# Create a pandas dataframe to visualize results
df_scan = pd.DataFrame(scan['hits'])

# Export DataFrame as a csv file
df_scan.to_csv('scan_output.csv', index=False)

# Example: only keep results with reliable uncertainty score and applicability domain score
df_scan = df_scan[(df_scan['potency_sigma']<0.8) & (df_scan['applicability_domain']>0.4)]

# Display the top entries of the dataframe
df_scan.head(10)

,uniprot_id,target_name,potency_mean,potency_sigma,alert,applicability_domain,gene_symbol,target_class,source
14,P35372,mu-type opioid receptor,6.244,0.789,False,0.438,OPRM1,GPCR,Bowes 2012
17,P41595,5-hydroxytryptamine receptor 2b,6.121,0.714,False,0.421,HTR2B,GPCR,Bowes 2012
28,Q08499,"3',5'-cyclic-amp phosphodiesterase 4d",5.919,0.749,False,0.405,PDE4D,Enzyme,Bowes 2012
33,P10275,androgen receptor,5.825,0.525,False,0.424,AR,Nuclear receptor,Bowes 2012
36,Q01959,sodium-dependent dopamine transporter,5.793,0.691,False,0.472,SLC6A3,Enzyme,Bowes 2012
37,P06239,tyrosine-protein kinase lck,5.542,0.551,False,0.679,LCK,Enzyme,Bowes 2012
39,Q12809,voltage-gated inwardly rectifying potassium ch...,5.327,0.409,False,0.613,KCNH2,Ion Channel,Bowes 2012
43,P22303,acetylcholinesterase,5.051,0.600,False,0.430,ACHE,Enzyme,Bowes 2012


### 2c. Screen (`mode:"screen"`)

Score a library of compounds against a target. Compounds are passed as a list of SMILES, which you can hardcode like in previous examples, or instead read from an existing `csv` file using a library such as `pandas` - see example below.

Please look at the schema file for this mode for comprehensive details.

In [10]:
# Example: we load some SMILES from a headerless csv file to use as inputs.
df = pd.read_csv('input_smiles_100k.csv', header=None)
some_smiles = df[0].tolist()[:10]

# Run screen
screen_payload = {
    'mode': 'screen',
    'instances': [{'smiles': some_smiles, 'uniprot_id': 'P00533'}],
}
result = invoke(screen_payload)

# Parse results
screen = result['predictions'][0]

In [11]:
# Create a pandas dataframe to visualize results
df = pd.DataFrame(screen['results'])

# A dataframe can be easily exported as a csv file
df.to_csv('screen_output.csv', index=False)

# Display a few entries of the dataframe
df.head(10)

,smiles,potency_mean,potency_sigma,applicability_domain
0,C[C@@H]1CN(c2ccc(CNC(=O)[C@H](C)n3cncn3)cn2)CCO1,6.187,0.648,0.308
1,NC(=O)CSc1ncnc2c1sc1nc(-c3ccccc3)c3c(c12)CCCC3,6.131,0.848,0.434
2,CS(=O)(=O)c1cnc(NCc2cnc(C(F)F)s2)nc1,5.704,0.959,0.250
3,Cc1c2cnn(-c3ccccc3)c(=O)c2c(C)n1CCCC(=O)NC[C@@...,5.634,0.496,0.435
4,C[C@@H]1CN([C@@H](C)C(=O)Nc2sccc2C(N)=O)CCO1,5.557,0.509,0.252
5,CCCCn1ccnc1[C@H]1CCCN(C(=O)Nc2ccccc2C)C1,5.519,0.584,0.371
6,Cc1n[nH]c(C)c1[C@H]1CCCN1Cc1cn2cccnc2n1,5.468,0.716,0.280
7,COc1ccc(-c2c(-c3oc(C)cc(=O)c3O)cnn2C(C)(C)C)cc1,5.274,0.618,0.398
8,CC(C)N1C[C@H](CN2CCCCCC2)CCC1=O,5.123,0.584,0.247
9,C[C@@H]1Sc2ccc(C(=O)NCc3nc(-c4ccccc4)n[nH]3)cc...,4.931,0.249,0.406


#### Multiple real-time calls to handle larger inputs

If you would like to run `screen` on a set of SMILES that exceeds the real-time cap (10000 at the time of writing), you can simply call `screen` repeatedly over successive "chunks" of SMILES. As long as the chunk size you set is below the cap, each call will return successfully.

The code below shows how you can go beyond the cap with no additional effort. We demonstrate on a small example to avoid incurring unecessary charges, but you could imagine an input file of size 200k SMILES and the chunk size set to 10k. 

For any input of size > 500k SMILES, you may want to consider a batch transform job instead (see corresponding section for advantages, such as parallelism, not monitoring needed, survive notebook disconnects ...).

In [12]:
import math

# Example: more SMILES
df = pd.read_csv('input_smiles_100k.csv', header=None)
many_smiles = df[0].tolist()[:22]

# Define target and chunk-size (chunk size can go up until the real-time cap, like 10000.)
uniprot_id = 'P00533'
chunk_size = 10

# Create an empty dataframe, populate it with results of successive real-time calls
for i in range(0, math.ceil(len(many_smiles)/chunk_size)):
    # Select next chunk of SMILES
    pos = i*chunk_size
    smiles_chunk = many_smiles[pos:min(pos+chunk_size, len(many_smiles))]

    # Run screen
    screen_payload = {'mode': 'screen', 'instances': [{'smiles': smiles_chunk, 'uniprot_id': uniprot_id}]}
    result = invoke(screen_payload)

    # Parse results, agglomerate into a single dataframe
    screen = result['predictions'][0]
    df_chunk = pd.DataFrame(screen['results'])
    df_screen_full = df_chunk if i == 0 else pd.concat([df_screen_full, df_chunk], ignore_index=True)

df_screen_full.head(len(many_smiles))

,smiles,potency_mean,potency_sigma,applicability_domain
0,C[C@@H]1CN(c2ccc(CNC(=O)[C@H](C)n3cncn3)cn2)CCO1,6.187,0.648,0.308
1,NC(=O)CSc1ncnc2c1sc1nc(-c3ccccc3)c3c(c12)CCCC3,6.131,0.848,0.434
2,CS(=O)(=O)c1cnc(NCc2cnc(C(F)F)s2)nc1,5.704,0.959,0.250
3,Cc1c2cnn(-c3ccccc3)c(=O)c2c(C)n1CCCC(=O)NC[C@@...,5.634,0.496,0.435
4,C[C@@H]1CN([C@@H](C)C(=O)Nc2sccc2C(N)=O)CCO1,5.557,0.509,0.252
5,CCCCn1ccnc1[C@H]1CCCN(C(=O)Nc2ccccc2C)C1,5.519,0.584,0.371
6,Cc1n[nH]c(C)c1[C@H]1CCCN1Cc1cn2cccnc2n1,5.468,0.716,0.280
7,COc1ccc(-c2c(-c3oc(C)cc(=O)c3O)cnn2C(C)(C)C)cc1,5.274,0.618,0.398
8,CC(C)N1C[C@H](CN2CCCCCC2)CCC1=O,5.123,0.584,0.247
9,C[C@@H]1Sc2ccc(C(=O)NCc3nc(-c4ccccc4)n[nH]3)cc...,4.931,0.249,0.406


### 2d. Selectivity (`mode:"selectivity"`)

Compute the fold selectivity on top of the potency scores for the target and a list of a few off-targets.
The fold selectivity is computed using the following formula: $$\large fold\_selectivity = 10^{({score_{target}} - {score_{off\text{-}target}})}$$

- A value of `100` means a 100-fold selectivity
- If the value is greater than `1`, it means the potency prediction for the target was higher than for the off-target (and lower than `1` means the contrary).


Please look at the schema file for this mode for comprehensive details.

In [16]:
selectivity_payload = {
    'mode': 'selectivity',
    'instances': [{'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
                   'uniprot_id': 'P00533',
                   'off_targets': ['P07550', 'P11362'],
                  }]
    }

result = invoke(selectivity_payload)
selectivity = result['predictions'][0]

# Parse result
print(json.dumps(selectivity, indent=2))

{
  "smiles": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
  "uniprot_id": "P00533",
  "target_name": "epidermal growth factor receptor",
  "potency_mean": 7.639,
  "potency_sigma": 1.115,
  "applicability_domain": 1.0,
  "fold_selectivity_against_strongest_off_target": 10.625,
  "off_targets": [
    {
      "off_target": "P11362",
      "off_target_name": "fibroblast growth factor receptor 1",
      "potency_mean": 5.667,
      "potency_sigma": 0.648,
      "applicability_domain": 0.844,
      "fold_selectivity": 93.903
    },
    {
      "off_target": "P07550",
      "off_target_name": "beta-2 adrenergic receptor",
      "potency_mean": 6.613,
      "potency_sigma": 0.724,
      "applicability_domain": 0.287,
      "fold_selectivity": 10.625
    }
  ]
}


In [17]:
# Create a pandas dataframe to visualize results
df = pd.DataFrame(selectivity['off_targets'])

# A dataframe can be easily exported as a csv file
df.to_csv('selectivity_output.csv', index=False)

# Display a few entries of the dataframe
df.head(10)

,off_target,off_target_name,potency_mean,potency_sigma,applicability_domain,fold_selectivity
0,P11362,fibroblast growth factor receptor 1,5.667,0.648,0.844,93.903
1,P07550,beta-2 adrenergic receptor,6.613,0.724,0.287,10.625


#### Many target / off-targets fold selectivity from a previous call to `scan`

Since fold selectivity can be computed from potency scores, you can reuse the output from a previous `scan` - which can cover the entire proteome in under a minute - to quickly compute fold selectivity for any target, against all off-targets in your `scan` results.

The example cell below selects an arbitrary target from the scan results of the previous section, and adds a new column to your dataframe using a hardcoded **fold selectivity formula of your choice**.

> ⚠️ The cell below may break depending on what panel you used in your last run of `scan` while running this notebook. If it does, pick a different uniprot ID featured in your results.

In [18]:
# Retrieve potency score for the desired target in a dataframe contain scan results
target = 'P35372'
ic50_target = df_scan.loc[df_scan['uniprot_id'] == target, 'potency_mean'].values[0]

# Add an extra fold selectivity column for the chosen target
df2 = df_scan.copy()
df2[f'fold_selectivity_{target}'] = 10**(ic50_target - df2['potency_mean'])
df2.head(10)

,uniprot_id,target_name,potency_mean,potency_sigma,alert,applicability_domain,gene_symbol,target_class,source,fold_selectivity_P35372
6,P08908,5-hydroxytryptamine receptor 1a,6.573,0.846,False,0.631,HTR1A,GPCR,Bowes 2012,0.468813
11,P14416,d,6.289,0.964,False,0.475,DRD2,GPCR,Bowes 2012,0.901571
14,P35372,mu-type opioid receptor,6.244,0.789,False,0.438,OPRM1,GPCR,Bowes 2012,1.000000
17,P41595,5-hydroxytryptamine receptor 2b,6.121,0.714,False,0.421,HTR2B,GPCR,Bowes 2012,1.327394
20,P31645,sodium-dependent serotonin transporter,6.018,0.840,False,0.472,SLC6A4,Enzyme,Bowes 2012,1.682674
28,Q08499,"3',5'-cyclic-amp phosphodiesterase 4d",5.919,0.749,False,0.405,PDE4D,Enzyme,Bowes 2012,2.113489
33,P10275,androgen receptor,5.825,0.525,False,0.424,AR,Nuclear receptor,Bowes 2012,2.624219
36,Q01959,sodium-dependent dopamine transporter,5.793,0.691,False,0.472,SLC6A3,Enzyme,Bowes 2012,2.824880
37,P06239,tyrosine-protein kinase lck,5.542,0.551,False,0.679,LCK,Enzyme,Bowes 2012,5.035006
39,Q12809,voltage-gated inwardly rectifying potassium ch...,5.327,0.409,False,0.613,KCNH2,Ion Channel,Bowes 2012,8.260379


## Real-time: clean up when you're done

**Run this when you are done to stop AWS infrastructure charges.**

In real-time, we are only charging you for the number of predictions you actually run, but as long as your endpoint is up, AWS is still charging you at the corresponding standard machine-hour rate. Depending on how long you leave it idle, it may add up to significant charges.

You may have inadvertently created a few models and endpoints by executing the cells in this notebook. On top of running this cell, we recommend you check for live endpoints in the AWS console for SageMaker, and terminate the relevant ones.

In [ ]:
# A separate command cleans up the model as well, after we're done with it at the end of this notebook.
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
print('Cleaned up endpoint and endpoint config')

Cleaned up endpoint, endpoint config, and models.


## 3. Batch transform: screening large compound libraries with `screen` (recommended for 500k and beyond)

[Batch transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html) runs large asynchronous jobs. Some key aspects:

- it is priced in machine hours (pro-rated per minute)
- returns the same predictions as real-time calculations
- a powerful tool to run very large input libraries in parallel, as it starts and kills machines on your behalf - even if the job takes several days, read/write files to AWS S3, survives any potential jupyter notebook disconnection, and you do not need to "babysit the job": you can go do something else
- you can monitor the job, check the parameters, logs, and see the input / output being gradually written to S3 buckets through the Web UI (SageMaker side menu > Batch transform jobs).

We provide the ability to run `screen` (single-target) on an arbitrarily large compound library `csv` file. Users can run the batch transform job by providing a `csv` residing in S3, and get back a `csv` output with scores. **We recommend using this tool for runs of size 500k SMILES and beyond, with a chunk size per invocation of 50k input pairs**. 

For smaller sizes, we recommend a loop over real-time calls to `screen`, which gives you a lower, more predictable cost with comparable / better time-to-results. Batch transform needs several minutes to start machines and bills you while the model is loading (~90s), which makes it less appealing for size < 500k. It really shines for large datasets.

Batch transform needs you to cut your input into "chunks" (e.g smaller `csv` files) that it distributes at runtime across your instances. Processing each chunk results in what SageMaker calls an "invocation". each invocation has a timeout of 60 minutes and can take as much as 100MB of data per input. You can set the desired number of machines using the variable `InstanceCount` in the example below. The output of each invocation is an output `csv` file written to S3, that users can agglomerate back into a single file with all results. This section shows how to perform these actions.

### Input files (csv)

In batch transform, both inputs and outputs are enclosed in `csv` files, instead of being passed as variables. This `screen` job requires:

- A single **headerless** input `csv` file, containing both SMILES and target information.
- Column 1 contains SMILES
- Column 2 contains a single target UniProt, and must be identical for each row.

First, we ensure a S3 bucket is available for the purpose of this example, and set up input and output paths for the data in our S3 buckets. You can instead reuse an existing S3 bucket with paths of your choosing.

In [19]:
# S3 bucket setup
account = boto3.client('sts', region_name=region).get_caller_identity()['Account']
bucket  = f'sagemaker-{region}-{account}'

try:
    s3_client.head_bucket(Bucket=bucket)
except s3_client.exceptions.ClientError:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket)
    else:
        s3_client.create_bucket(Bucket=bucket, CreateBucketConfiguration={'LocationConstraint': region})

You can define yourself the chunk size. The following code will split your `csv` into chunks of that size. Those chunks will be distributed across the instances started by your batch transform job (ensure the number of chunks ≥ number of machines).

The example below loads a "small library" to avoid incurring unecessary charges for the sake of demonstration, picture instead a library containing several millions SMILES.

In [20]:
# Input file name
fname = 'input_smiles_15k.csv'
fname_prefix = fname.split('.')[0]

# Set chunk size as you please, same for input and output S3 folder names
rows_per_file = 10000
timestamp = int(time.time())
s3_input_folder  = f'aqpotency/screen-batch-input/{timestamp}'
s3_output_folder = f'aqpotency/screen-batch-output/{timestamp}'

input_s3  = f's3://{bucket}/{s3_input_folder}'
output_s3 = f's3://{bucket}/{s3_output_folder}'

# Read and split the full csv file into chunks, writes individual chunk csv files to S3
for i, chunk in enumerate(pd.read_csv(fname, chunksize=rows_per_file, header=None)):
    fname_chunk = f'{fname_prefix}_{i+1}.csv'
    chunk.to_csv(fname_chunk, index=False, header=False)
    s3_client.upload_file(fname_chunk, bucket, f'{s3_input_folder}/{fname_chunk}')
print(f'{i+1} files uploaded to {input_s3}')

2 files uploaded to s3://sagemaker-us-east-1-769953010191/aqpotency/screen-batch-input/1790649336


You can now launch the batch transform job:

In [21]:
instance_count = 1      # Increase to parallelize over input csv files in the S3 input folder
job_name = f'aqpotency-screen-{timestamp}'

sagemaker_client.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,                 # subscribed model from section 3
    BatchStrategy='MultiRecord',
    MaxPayloadInMB=6,
    TransformInput={
        'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': input_s3}},
        'ContentType': 'text/csv',
        'SplitType': 'Line',              # mini-batch within each file
    },
    TransformOutput={'S3OutputPath': output_s3, 'Accept': 'text/csv', 'AssembleWith': 'Line'},
    TransformResources={'InstanceType': 'ml.m5.2xlarge', 'InstanceCount': instance_count},
)
print('Submitted batch job:', job_name)
print('Results will be written to:', output_s3)

Submitted batch job: aqpotency-screen-1790649336
Results will be written to: s3://sagemaker-us-east-1-769953010191/aqpotency/screen-batch-output/1790649336


The job has been submitted asynchronously and its status can be tracked through AWS console for SageMaker ("Jobs" menu).
The next cell blocks until the job is complete. Once that's done, we can retrieve the results. It should take about 5 minutes for this small job - most of the time is spent starting the endpoint and loading the model in this small job, batch becomes really interesting for larger jobs.

In [22]:
print('Waiting for the batch job to finish -- 5 minutes for the base example')
sagemaker_client.get_waiter('transform_job_completed_or_stopped').wait(TransformJobName=job_name)
print('DONE. Output prefix:', output_s3)
response = sagemaker_client.describe_transform_job(TransformJobName=job_name)
print(f'Job {response["TransformJobName"]} completed in {(response["TransformEndTime"]-response["TransformStartTime"]).total_seconds()/60:.1f} minutes.')

Waiting for the batch job to finish -- 5 minutes for the base example
DONE. Output prefix: s3://sagemaker-us-east-1-769953010191/aqpotency/screen-batch-output/1790649336
Job aqpotency-screen-1790649336 completed in 4.5 minutes.


We can recombine all files in the output directory to get a single `csv` back. They have the same name as the input file except they have a `.out` extension.

In [28]:
# List all .out files in S3 output directory
objs = s3_client.list_objects_v2(Bucket=bucket, Prefix=s3_output_folder).get('Contents', [])
output_files = sorted(o['Key'] for o in objs if o['Key'].endswith('.out'))

# Combines all of them into a single dataframe
cols = ['SMILES', 'uniprot_id', 'potency_mean', 'potency_sigma', 'applicability_domain']
records = []
for k in output_files:
    text  = s3_client.get_object(Bucket=bucket, Key=k)['Body'].read().decode().strip()
    lines = [ln for ln in text.split('\n') if ln]
    records += [ln.split(',') for ln in lines]
df = pd.DataFrame(records, columns=cols)
print(f'\nCombined {len(df)} rows from {len(output_files)} file(s).')

# Cast score columns as floats
for col_name in ['potency_mean', 'potency_sigma', 'applicability_domain']:
    df[col_name] = df[col_name].astype(float)

# Sort final output file and write locally, display first few rows
df = df.sort_values('potency_mean', ascending=False)
df.to_csv(f'{fname_prefix}_output_full.csv', index=False)
df.head(10)


Combined 15000 rows from 2 file(s).


,SMILES,uniprot_id,potency_mean,potency_sigma,applicability_domain
1663,CN(C)C1CCN(c2ccc(Nc3cc4c(N5CCC(CO)CC5)nc(Nc5cc...,P00533,7.902,0.637,1.000
12583,Cc1ccc(NC(=O)N2CC[C@@H](CC(F)(F)F)C2)cc1-c1cc(...,P00533,7.402,0.600,0.392
14872,CN(C)c1nc(Cl)ncc1Br,P00533,7.365,1.428,0.294
1411,CCOc1c(F)cc([C@H](CCC(C)C)N2CCNCC2)cc1Cl,P00533,7.330,1.147,0.281
9132,CC(C)n1ncc2c1CCC[C@H]2Nc1cnn2c1OCC(C)(C)C2,P00533,7.277,1.154,0.243
129,Fc1cc([C@H]2OCC[C@@H]2Nc2ncnc3[nH]ccc23)ccc1Cl,P00533,7.272,0.925,0.506
1086,CCn1cc(S(=O)(=O)N2C[C@H]3CN(c4nc5ccccc5n4C)C[C...,P00533,7.215,1.224,0.331
3317,Fc1cccc(CNc2nc(Cl)ncc2Br)c1,P00533,7.211,1.169,0.413
8267,CCc1ccc(OCc2ccc(-c3nc4c5c(ncn4n3)Oc3n[nH]c(C)c...,P00533,7.210,0.543,0.358
12299,CCn1nc(C(C)C)cc1C(=O)N1CCC[C@@H](c2ccn[nH]2)C1,P00533,7.200,0.871,0.303


As usual, you can leverage all available `pandas` and spreadsheet tools to sort, filter, plot and perform any steps of your data analysis.

We finish this notebook by cleaning up and removing the model object that was used for both real-time and batch cells.

In [ ]:
sagemaker_client.delete_model(ModelName=model_name)